<h2> This code uses Stable baselines3 to train the model instead of mjrl </h2>

Import modules

In [1]:
import myosuite
import gym
import skvideo.io
import numpy as np
import os
from stable_baselines3.common.logger import configure
from IPython.display import HTML
from base64 import b64encode
from sb3_contrib import RecurrentPPO
from stable_baselines3 import PPO
import torch
 
def show_video(video_path, video_width = 400):
   
  video_file = open(video_path, "r+b").read()
 
  video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"
  return HTML(f"""<video autoplay width={video_width} controls><source src="{video_url}"></video>""")


tmp_path = "/tmp"
# set up logger
new_logger = configure(tmp_path, ["stdout", "csv", "tensorboard"])

MyoSuite:> Registering Myo Envs
pygame 2.5.2 (SDL 2.28.3, Python 3.8.18)
Hello from the pygame community. https://www.pygame.org/contribute.html
Logging to /tmp


# Vectorize environment

In [2]:
from stable_baselines3.common.vec_env import DummyVecEnv
import gym

env = gym.make('CenterReachOut-v0')

    MyoSuite: A contact-rich simulation suite for musculoskeletal motor control
        Vittorio Caggiano, Huawei Wang, Guillaume Durandau, Massimo Sartori, Vikash Kumar
        L4DC-2019 | https://sites.google.com/view/myosuite
    


# MLP Training

In [ ]:
# Setup logging directory
logdir = "logs"

model = PPO(
    "MlpPolicy",
    env,
    ent_coef=0.01,
    tensorboard_log=logdir,
    batch_size=32,
    n_steps=12,
    device="cpu",
    policy_kwargs={
        "net_arch": [dict(pi=[64, 64], vf=[64, 64])],
        "activation_fn": torch.nn.Sigmoid,
    },
    
)
print(f"Using device: {model.device}")

Using device: cpu


c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:77: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")
c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\ppo\ppo.py:155: UserWarning: You have specified a mini-batch size of 32, but because the `RolloutBuffer` is of size `n_steps * n_envs = 12`, after every 0 untruncated mini-batches, there will be a truncated mini-batch of size 12
We recommend

<h3> RNN training </h3>

In [ ]:
# Setup logging directory
logdir = "logs"

model = RecurrentPPO(
    "MlpLstmPolicy",
    env,
    ent_coef=0.01,
    tensorboard_log=logdir,
    batch_size=1000,
    n_steps=20,
    device="cpu",
    policy_kwargs=dict(lstm_hidden_size=256, net_arch=[dict(pi=[256], vf=[256])]),  
)
print(f"Using device: {model.device}")

Using device: cpu


c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\vec_env\patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\vec_env\base_vec_env.py:77: UserWarning: The `render_mode` attribute is not defined in your environment. It will be set to None.
  warnings.warn("The `render_mode` attribute is not defined in your environment. It will be set to None.")
c:\Users\Elyas\anaconda3\envs\myosuite\lib\site-packages\stable_baselines3\common\policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  wa

In [ ]:
if not os.path.exists(logdir):
    os.makedirs(logdir)

model.learn(total_timesteps=600000)

In [5]:
model.save('test') #Saves model 

<h2> Load sb3 model <h2>

In [ ]:
model = PPO.load("test", env=env, device="cuda")

# Render trained policy
frames = []
for _ in range(20): # 5 random targets
  env.reset()
  ep_rewards = []
  done = False
  obs = env.reset()
  while not done:
      frame = env.sim.renderer.render_offscreen(camera_id=1,
                        width=400,
                        height=400)
      frames.append(frame)
      o = env.get_obs()
      # get the next action from the policy
      action, _ = model.predict(o)
      #print(pi.show_activations())
      #print(pi.get_neurons())
      # take an action based on the current observation
      obs, reward, done, info = env.step(action)

env.close()

In [ ]:
import imageio

os.makedirs('videos', exist_ok=True)
video_path = 'videos/test.mp4'
# make a local copy
imageio.mimsave(video_path, frames, fps=30)
show_video('videos/test.mp4')